In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import Dataset, DataLoader

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/digit-recognizer'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, train_size=0.8, random_state=42)
train_df=train_df.to_numpy()
test_df=test_df.to_numpy()
y_train, x_train = train_df[:,0], train_df[:,1:].astype(np.float32)
y_test, x_test = test_df[:,0], test_df[:,1:].astype(np.float32)

In [ ]:
x_train,y_train,x_test,y_test=map(torch.tensor,(x_train,y_train,x_test,y_test))

In [ ]:
eps = 1e-6
def relu(x):
    return x.clamp(min=0)


def batch_norm(x, gamma):
    batch_mean = x.mean(dim=0)
    batch_var = (x-batch_mean).var(dim=0)
    x_norm = (x - batch_mean) / torch.sqrt(batch_var + eps)
    return gamma * x_norm

In [ ]:
BATCH_SIZE=64
epochs=300
EPOCHS_PER_EVAL=10
lr = 1e-2
hidden_features=512

In [ ]:
x_train,y_train,x_test,y_test=[t.to(device) for t in [x_train,y_train,x_test,y_test]]
w1 = torch.distributions.Uniform(-np.sqrt(1/784),np.sqrt(1/784)).sample((hidden_features,784)).to(device).requires_grad_()
b1 = torch.distributions.Uniform(-np.sqrt(1/784),np.sqrt(1/784)).sample((hidden_features,)).to(device).requires_grad_()
w2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features),np.sqrt(1/hidden_features)).sample((hidden_features,hidden_features)).to(device).requires_grad_()
b2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features),np.sqrt(1/hidden_features)).sample((hidden_features,)).to(device).requires_grad_()
w3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features),np.sqrt(1/hidden_features)).sample((10,hidden_features)).to(device).requires_grad_()
b3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features),np.sqrt(1/hidden_features)).sample((10,)).to(device).requires_grad_()

In [ ]:
def calc_loss(x, y_vals,accuracy=False):
    x = x/255.0
    a1 = relu(x@w1.T+b1)
    a2 = relu(a1@w2.T+b2)
    z = (a2@w3.T+b3)
    log_softmax = z - torch.logsumexp(z, dim=-1, keepdim=True)
    loss = -log_softmax[range(len(y_vals)),y_vals].mean()
    if accuracy:
        return loss,(torch.argmax(z,dim=-1)==y_vals).float().mean()
    else:
        return loss

for epoch in range(epochs):
    perm = torch.randperm(len(x_train))
    x_train = x_train[perm]
    y_train = y_train[perm]
    for i in range(0, len(x_train), BATCH_SIZE):
        x = x_train[i:i+BATCH_SIZE]
        y_vals = y_train[i:i+BATCH_SIZE]
    
        loss = calc_loss(x, y_vals)
        loss.backward()
        with torch.no_grad():
            w1 -= w1.grad * lr
            b1 -= b1.grad * lr
            w2 -= w2.grad * lr
            b2 -= b2.grad * lr
            w3 -= w3.grad * lr
            b3 -= b3.grad * lr
            w1.grad.zero_()
            b1.grad.zero_()
            w2.grad.zero_()
            b2.grad.zero_()
            w3.grad.zero_()
            b3.grad.zero_()


    if epoch%EPOCHS_PER_EVAL==0:
        with torch.no_grad():
            loss,accuracy = calc_loss(x_test, y_test, accuracy=True)
            print(f"loss = {loss} accuracy = {accuracy}")

In [ ]:
class MNIST(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.n = len(x)
    def __len__(self):
        return self.n
    def __getitem__(self,idx):
        return self.x[idx], self.y[idx]

train_ds = MNIST(x_train, y_train)
test_ds = MNIST(x_test,y_test)
train_dl = DataLoader(train_ds,batch_size=BATCH_SIZE)
test_dl = DataLoader(test_ds,batch_size=BATCH_SIZE)

In [ ]:
class MLP(nn.Module):
    def __init__(self,in_dim,hidden_dim,out_dim):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(in_dim,hidden_dim),nn.ReLU(),
                                 nn.Linear(hidden_dim,hidden_dim),nn.ReLU(),
                                 nn.Linear(hidden_dim,out_dim))
    def forward(self, x):
        return self.mlp(x/255.0)

model = MLP(784,512,10).to(device)
loss = nn.CrossEntropyLoss()
opt = optim.SGD(model.parameters(), lr = lr)
for epoch in range(epochs):
    model.train()
    for xb,yb in train_dl:
        loss(model(xb),yb).backward()
        opt.step()
        opt.zero_grad()
    model.eval()
    with torch.no_grad():
        if epoch%EPOCHS_PER_EVAL==0:
            out = model(x_test)
            print(f"loss = {loss(out,y_test)} accuracy = {(torch.argmax(out,dim=-1)==y_test).float().mean()}")